In [1]:
import os

# os.environ['GITHUB_ACCESS_TOKEN'] = ''

# print(os.getenv("GITHUB_ACCESS_TOKEN", None))

from lean_dojo import *

In [2]:
# Repo containing both mathlib4 and Math-in-lean, fetched in March 2024
# Pickled at '/home/mcwave/code/automath/atp/datasets/traced_repo_math_in_lean.pkl'
# repo = LeanGitRepo(
#     "https://github.com/xiaoxin-yin/math-in-lean",
#     "20077bcd4392317ddb9605404fda3a85e40e8956"
# )

# Repo Mathlib4 on 6/17/24
repo = LeanGitRepo(
    "https://github.com/leanprover-community/mathlib4",
    "27c6744e1c0e25d676be5eb252cd4b6d30c6acc7",
)

repo

LeanGitRepo(url='https://github.com/leanprover-community/mathlib4', commit='27c6744e1c0e25d676be5eb252cd4b6d30c6acc7')

In [58]:
# import pickle

# fout = open('/home/mcwave/code/automath/atp/datasets/traced_repo_math_in_lean.pkl', 'wb')
# pickle.dump(traced_repo, fout)
# fout.close()

In [3]:
# import pickle

# fin = open('/home/mcwave/code/automath/atp/datasets/traced_repo_mathlib4_20240617.pkl', 'rb')
# traced_repo = pickle.load(fin)
# fin.close()

In [ ]:
# import pickle

# fin = open('/home/mcwave/code/automath/atp/datasets/train_traced_theorems_repo_math_in_lean.pkl', 'rb')
# train_traced_theorems = pickle.load(fin)
# fin.close()

In [4]:
# Save traced_theorems to a pickle file since they are only data we need

import json

def generate_train_file_theorems(train_tac_templates_path, output_path):
    fin = open(train_tac_templates_path, 'r')
    train_tac_templates = json.load(fin)
    fin.close()
    #
    train_file_theorems = {}
    for i in range(len(train_tac_templates)):
        full_name = train_tac_templates[i][2]
        file_path = train_tac_templates[i][3]
        if file_path not in train_file_theorems:
            train_file_theorems[file_path] = set()
        train_file_theorems[file_path].add(full_name)
    #
    train_traced_theorems = dict()
    for file_path, full_names in train_file_theorems.items():
        traced_file = traced_repo.get_traced_file(file_path)
        premises = traced_file.get_premise_definitions()
        results = []
        for premise in premises:
            full_name = premise['full_name']
            if full_name in full_names:
                thm = traced_file.get_traced_theorem(full_name)
                thm.comments.insert(0, premise['code'])
                if file_path not in train_traced_theorems:
                    train_traced_theorems[file_path] = dict()
                train_traced_theorems[file_path][full_name] = thm
    #
    for file_path, full_names in train_file_theorems.items():
        if file_path not in train_traced_theorems:
            print(file_path, "not found!")
            continue
        for full_name in full_names:
            if full_name not in train_traced_theorems[file_path]:
                print(full_name, "not found in", file_path)
    #
    fout = open(, 'wb')
    pickle.dump(train_traced_theorems, fout)
    fout.close()


#train_tac_templates_path = '/home/mcwave/code/automath/atp/datasets/tac_templates_in_files/train_tac_templates.json'
#output_path = '/home/mcwave/code/automath/atp/datasets/train_traced_theorems_repo_math_in_lean.pkl'
#generate_train_file_theorems(train_tac_templates_path, output_path)


SyntaxError: invalid syntax (3179443656.py, line 40)

In [2]:
# Exclude processed theoresm from train_traced_theorems
import os
import pickle
from lean_dojo import *

def get_all_theorems_processed(folder_paths, verbose=True):
    # Enumerate all .pkl files in the folder
    output = {}
    for folder_path in folder_paths:
        for filename in os.listdir(folder_path):
            if filename.endswith(".pkl"):
                found = False
                file_path = os.path.join(folder_path, filename)
                fin = open(file_path, 'rb')
                while True:
                    try:
                        result = pickle.load(fin)
                    except Exception as e:
                        break
                    file_path, full_name, theorem, state_pair = result
                    if file_path not in output:
                        output[file_path] = set()
                    output[file_path].add(full_name)
                    found = True
                fin.close()
                if verbose and found:
                    print(filename, len(output[file_path]), "theorems")
    return output

fin = open('/home/mcwave/code/automath/atp/datasets/train_traced_theorems_repo_math_in_lean.pkl', 'rb')
train_traced_theorems = pickle.load(fin)
fin.close()
    
previous_output_paths = [
    '/home/mcwave/code/automath/atp/datasets/provability/rag-20240710',
    '/home/mcwave/code/automath/atp/datasets/provability/rag-20240712'
]

print("Getting previously computed theorems")
processed_theorems = get_all_theorems_processed(previous_output_paths)
remaining_traced_theorems = dict()
for train_file_path, traced_theorems in train_traced_theorems.items():
    if train_file_path not in processed_theorems:
        remaining_traced_theorems[train_file_path] = traced_theorems
        continue
    remaining_theorems = dict()
    for full_name, thm in traced_theorems.items():
        if full_name not in processed_theorems[train_file_path]:
            remaining_theorems[full_name] = thm
        else:
            remaining_theorems = dict()
    print(train_file_path, len(traced_theorems), "->", len(remaining_theorems))
    remaining_traced_theorems[train_file_path] = remaining_theorems

fout = open('/home/mcwave/code/automath/atp/datasets/remaining_traced_theorems_repo_math_in_lean_20240617.pkl', 'wb')
pickle.dump(remaining_traced_theorems, fout)
fout.close()

Getting previously computed theorems
Mathlib__Algebra__BigOperators__Pi.lean.pkl 13 theorems
Mathlib__CategoryTheory__Iso.lean.pkl 46 theorems
Mathlib__Algebra__Order__Group__Int.lean.pkl 4 theorems
.lake__packages__lean4__src__lean__Init__Grind__Norm.lean.pkl 12 theorems
Mathlib__Order__IsWellOrderLimitElement.lean.pkl 11 theorems
.lake__packages__lean4__src__lean__Init__Data__Nat__MinMax.lean.pkl 13 theorems
Mathlib__Data__Nat__Upto.lean.pkl 1 theorems
.lake__packages__batteries__Batteries__Control__ForInStep__Lemmas.lean.pkl 11 theorems
Mathlib__CategoryTheory__Limits__KanExtension.lean.pkl 2 theorems
Mathlib__Order__Extension__Linear.lean.pkl 1 theorems
Mathlib__Data__Finset__Pointwise.lean.pkl 76 theorems
Mathlib__MeasureTheory__Group__GeometryOfNumbers.lean.pkl 1 theorems
Mathlib__LinearAlgebra__InvariantBasisNumber.lean.pkl 10 theorems
Mathlib__Data__Int__SuccPred.lean.pkl 9 theorems
Mathlib__Geometry__Manifold__Sheaf__Smooth.lean.pkl 6 theorems
Mathlib__MeasureTheory__Integral_

Mathlib__Order__Chain.lean.pkl 42 theorems
.lake__packages__batteries__Batteries__Logic.lean.pkl 2 theorems
Mathlib__MeasureTheory__Function__EssSup.lean.pkl 41 theorems
.lake__packages__batteries__Batteries__Data__ByteArray.lean.pkl 30 theorems
Mathlib__Data__Sign.lean.pkl 49 theorems
.lake__packages__lean4__src__lean__Init__Data__Nat__Bitwise__Lemmas.lean.pkl 52 theorems
Mathlib__SetTheory__Game__State.lean.pkl 4 theorems
Mathlib__LinearAlgebra__Contraction.lean.pkl 6 theorems
Mathlib__NumberTheory__Zsqrtd__QuadraticReciprocity.lean.pkl 3 theorems
Mathlib__RingTheory__Bezout.lean.pkl 3 theorems
Mathlib__Data__Real__ENatENNReal.lean.pkl 15 theorems
.lake__packages__batteries__Batteries__Control__Lemmas.lean.pkl 4 theorems
Mathlib__Algebra__Module__Submodule__IterateMapComap.lean.pkl 3 theorems
Mathlib__MeasureTheory__Measure__Dirac.lean.pkl 17 theorems
Mathlib__CategoryTheory__Preadditive__HomOrthogonal.lean.pkl 4 theorems
Mathlib__CategoryTheory__Sites__EpiMono.lean.pkl 1 theorems
Ma

Mathlib__Data__PFunctor__Multivariate__W.lean.pkl 14 theorems
Mathlib__Data__Set__Pairwise__Basic.lean.pkl 67 theorems
Mathlib__Topology__Homotopy__Path.lean.pkl 21 theorems
Mathlib__Data__Multiset__Basic.lean.pkl 456 theorems
Mathlib__Analysis__Normed__Field__InfiniteSum.lean.pkl 8 theorems
Mathlib__Combinatorics__Young__SemistandardTableau.lean.pkl 10 theorems
Mathlib__Topology__Homotopy__Product.lean.pkl 9 theorems
Mathlib__Order__Heyting__Hom.lean.pkl 47 theorems
Mathlib__Data__Stream__Init.lean.pkl 129 theorems
Mathlib__Analysis__SpecialFunctions__Complex__Log.lean.pkl 40 theorems
Mathlib__Algebra__Lie__Classical.lean.pkl 14 theorems
Mathlib__Algebra__Homology__ShortComplex__ConcreteCategory.lean.pkl 12 theorems
Mathlib__GroupTheory__GroupAction__Units.lean.pkl 7 theorems
Mathlib__GroupTheory__Perm__DomMulAct.lean.pkl 5 theorems
Mathlib__Algebra__Homology__SingleHomology.lean.pkl 20 theorems
Mathlib__AlgebraicGeometry__PrimeSpectrum__Basic.lean.pkl 18 theorems
Mathlib__MeasureTheo

In [40]:
traced_file = traced_repo.get_traced_file('.lake/packages/mathlib/Mathlib/Data/Set/Sups.lean')

print(traced_file)

NameError: name 'traced_repo' is not defined

In [5]:
traced_file.get_premise_definitions()

[{'full_name': 'IsCoprime',
  'code': 'def IsCoprime : Prop :=\n  ∃ a b, a * x + b * y = 1',
  'start': [36, 1],
  'end': [40, 27],
  'kind': 'commanddeclaration'},
 {'full_name': 'IsCoprime.symm',
  'code': '@[symm]\ntheorem IsCoprime.symm (H : IsCoprime x y) : IsCoprime y x',
  'start': [45, 1],
  'end': [48, 30],
  'kind': 'commanddeclaration'},
 {'full_name': 'isCoprime_comm',
  'code': 'theorem isCoprime_comm : IsCoprime x y ↔ IsCoprime y x',
  'start': [51, 1],
  'end': [52, 35],
  'kind': 'commanddeclaration'},
 {'full_name': 'isCoprime_self',
  'code': 'theorem isCoprime_self : IsCoprime x x ↔ IsUnit x',
  'start': [55, 1],
  'end': [58, 41],
  'kind': 'commanddeclaration'},
 {'full_name': 'isCoprime_zero_left',
  'code': 'theorem isCoprime_zero_left : IsCoprime 0 x ↔ IsUnit x',
  'start': [61, 1],
  'end': [64, 40],
  'kind': 'commanddeclaration'},
 {'full_name': 'isCoprime_zero_right',
  'code': 'theorem isCoprime_zero_right : IsCoprime x 0 ↔ IsUnit x',
  'start': [67, 1],
  

In [4]:
thm = traced_file.get_traced_theorem("Solutions_S01_Calculating_ex2")

print(thm.theorem)

proof_node = thm.get_proof_node()
proof = proof_node.lean_file[proof_node.start : proof_node.end]
print(proof)

traced_tactics = thm.get_traced_tactics()
print("\ntraced_tactics:\n", traced_tactics)

tac = traced_tactics[0]
print("\ntac:\n", tac)

NameError: name 'traced_file' is not defined

In [2]:
from utils.lean_math_utils import *

#theorem = Theorem(repo, "MIL/C02_Basics/solutions/Solutions_S04_More_on_Order_and_Divisibility.lean",
#                 "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex1")

theorem = Theorem(repo, "MIL/C02_Basics/solutions/Solutions_S01_Calculating.lean",
                     "Solutions_S01_Calculating_ex2")

dojo, state_0 = Dojo(theorem).__enter__()

NameError: name 'Theorem' is not defined

In [11]:
state_1 = dojo.run_tac(state_0, "rw [← mul_assoc]")

#print(state_1)
print(state_1.pp)
classify_lean_elements(state_1.pp)

a b c : ℝ
⊢ a * b * c = b * (a * c)


({'a': 'variable', 'b': 'variable', 'c': 'variable'},
 {'a': 'ℝ', 'b': 'ℝ', 'c': 'ℝ'})

In [12]:
state_2 = dojo.run_tac(state_1, "rw [mul_comm a]") 

print(state_2)

TacticState(pp='a b c : ℝ\n⊢ b * a * c = b * (a * c)', id=2, message='')


In [38]:
tacs = ['rw [isCoprime_comm] at H H ⊢',
   'rw [mul_comm] at H',
   'rw [isCoprime_comm] at H H ⊢',
   'rw [isCoprime_comm] at H ⊢',
   'rw [isCoprime_comm] at H H ⊢',
   'intro y nvar0 nvar1 nvar2 nvar3',
   'rw [isCoprime_comm] at H H ⊢',
   'exact H.of_mul_left_left']

file_path = '.lake/packages/mathlib/Mathlib/Algebra/Ring/Commute.lean'
full_name = 'IsCoprime.of_mul_right_left'

theorem = Theorem(repo, file_path, full_name)

dojo, state_0 = Dojo(theorem).__enter__()
curr_state = state_0

for tac in tacs:
    print()
    print(tac)
    next_state = dojo.run_tac(curr_state, tac)
    print(next_state.pp)

NameError: name 'repo' is not defined

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

class TripletDataset(Dataset):
    def __init__(self, triplets, tokenizer):
        self.triplets = triplets
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, index):
        q, s, n = self.triplets[index]
        q_encoded = self.tokenizer(q, padding='max_length', max_length=256, truncation=True, return_tensors='pt')
        s_encoded = self.tokenizer(s, padding='max_length', max_length=64, truncation=True, return_tensors='pt')
        n_encoded = self.tokenizer(n, padding='max_length', max_length=64, truncation=True, return_tensors='pt')
        return q_encoded, s_encoded, n_encoded

class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, q_embed, s_embed, n_embed):
        dist_pos = torch.norm(q_embed - s_embed, p=2, dim=1)
        dist_neg = torch.norm(q_embed - n_embed, p=2, dim=1)
        loss = torch.mean(torch.relu(dist_pos - dist_neg + self.margin))
        return loss

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

model_state = torch.load('/home/mcwave/code/automath/atp/datasets/rag_tactic_templates/bert_embeder_state-batch64-60k-loss0047.model')
# model_tac   = torch.load('/home/mcwave/code/automath/atp/datasets/rag_tactic_templates/bert_embeder_tac-batch64-60k-loss0047.model')
device = model_state.device
print(device)

cuda:0


In [6]:
import json

import faiss

index = faiss.read_index('/home/mcwave/code/automath/atp/datasets/rag_tactic_templates/faiss_index_bert_embeds-batch64-60k-loss0047.idx')

fin = open('/home/mcwave/code/automath/atp/datasets/rag_tactic_templates/tac_template_freq.json', 'r')
tac_template_freq = json.load(fin)
fin.close()

all_tacs = list(tac_template_freq.keys())

In [7]:
# Example query string

def get_similar_tacs(query_string, model_state, tokenizer, faiss_index, all_tacs, num_returned=100, verbose=False):
    # Convert query string to BERT embedding
    device = model_state.device
    with torch.no_grad():
        encoded_input = tokenizer(query_string, padding=True, truncation=True, return_tensors='pt').to(device)
        outputs = model_state(**encoded_input)
        query_embedding = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()

    # Perform nearest neighbor search
    distances, indices = faiss_index.search(query_embedding, num_returned)

    if verbose:
        # Print the nearest neighbors and their distances
        print(f"Nearest neighbors for: {query_string}")
    output = []
    for i in range(num_returned):
        idx = indices[0][i]
        distance = distances[0][i]
        similarity = 1.0 / (1 + distance*distance/1000)
        tac = all_tacs[idx]
        output.append((all_tacs[idx],similarity))
        if verbose:
            print(f"{i+1}. {all_tacs[idx]} (similarity: {similarity})")
    return sorted(output, key=lambda item: item[1], reverse=True)

code = 'theorem Solutions_S01_Calculating_ex2 (a b c : ℝ) : a * (b * c) = b * (a * c)'
state_pp = state_0.pp
query = code + ' # ' + state_pp
print("QUERY:\n", query)

suggestions = get_similar_tacs(query, model_state, tokenizer, index, all_tacs, num_returned=200)
for i in range(len(suggestions)):
    print(suggestions[i])
#     if 'dvd_mul_of_dvd_left' in suggestions[i][0] :
#         print("Found at", i, ":", suggestions[i][0])

QUERY:
 theorem Solutions_S01_Calculating_ex2 (a b c : ℝ) : a * (b * c) = b * (a * c) # α : Type u
β : Type v
γ : Type w
R : Type x
inst✝ : Distrib R
a b c : R
⊢ Commute a b → Commute a c → Commute a (b + c)
('rintro rfl', 0.011596429749190968)
('ext', 0.011372392823415971)
('dsimp', 0.011303530163117005)
('exact', 0.011213295183600626)
('congr', 0.011120849577303457)
('split', 0.011099556780823295)
('exfalso', 0.01109826814126812)
('linarith', 0.01109768503157248)
('intros', 0.011035022142892674)
('intro {variable} {nvar0}', 0.011031285587746036)
('ext {nvar0} {nvar1}', 0.011018347528494265)
('constructor', 0.011012954331687471)
('aesop', 0.010974487317822438)
('intro {nvar0} {nvar1} {nvar2}', 0.010943911513003462)
('simp', 0.010898529243478592)
('symm', 0.010889929274163244)
('classical', 0.010888006223176056)
('contradiction', 0.010836437521164918)
('intro {nvar0}', 0.010827270273450323)
('calc', 0.01082434764489679)
('intro {nvar0} {nvar1} {nvar2} {nvar3} {nvar4}', 0.01081932920497

In [ ]:
import pickle

print("Loading traced theorems ...")
fin = open('/home/mcwave/code/automath/atp/datasets/traced_repo_mathlib4_20240617.pkl', 'rb')
train_traced_theorems = pickle.load(fin)
fin.close()

In [24]:
import lean_dojo
import random
from random import randint
import time

from utils.lean_math_utils import *

MAX_STEPS = 100000
MAX_TACTIC_FROM_TEMPLATE = 50
PENALTY_SEEN_TARGET_MULTIPLIER = 3

#theorem = Theorem(repo, "MIL/C02_Basics/solutions/Solutions_S05_Proving_Facts_about_Algebraic_Structures.lean",
#                  "Solutions_S05_Proving_Facts_about_Algebraic_Structures_ex1")
#theorem = Theorem(repo, "MIL/C02_Basics/solutions/Solutions_S01_Calculating.lean",
#                  "Solutions_S01_Calculating_ex2")

MAX_MEMORY_USAGE = 16*1024*1024*1024
MAX_STEPS = 100000
MAX_TACTIC_FROM_TEMPLATE = 50
PENALTY_SEEN_TARGET_MULTIPLIER = 3
MAX_NUM_DOJO_ATTEMPT = 2



# Print the trace from state_0 to state
def get_tactic_trace(curr_state, state_dict):
    _, parent_states, tactics = state_dict[curr_state]
    if len(parent_states) == 0:
        return []
    else:
        return get_tactic_trace(parent_states[0], state_dict) + [(tactics[0], curr_state)]
    
# Create the inverse tactic for a 'rw' tactic template
def get_inverse_tactic(tac_template):
    if not tac_template.startswith('rw') or not '[' in tac_template \
        or '←' in tac_template or '[{' in tac_template:
        return None
    result = tac_template.replace('[', '[← ')
    if ',' in result:
        pos = result.find(',')
        result = result[0:pos] + ']'
    return result

@timeout(1)
def dojo_run_tac(dojo, state, tactic):
    #     # Get the current process object
    #     process = psutil.Process(os.getpid())
    #     # Memory info (in bytes)
    #     mem_info = process.memory_info()
    #     # Print the RSS (Resident Set Size): total physical memory used
    #     print(f"Current memory RSS usage: {mem_info.rss / (1024 ** 2):.2f} MB", randint(0,100), end="")
    #print("Run tac", end="")
    result = dojo.run_tac(state, tactic)
    #print("Done")
    return result

# The complexity of a state if calculated from lengths of its targets
# seen_target_freq is a dict{string, int} of strings containing lines of targets that have been seen
# If all targets in the current state has been seen, add to complexity PENALTY_SEEN_TARGET_MULTIPLIER*min_freq
# At the same time, seen_target_freq is updated by the targets in this state
# if base_complexity is not none, complexity is base_complexity+1
def explore_state_complexity(state, base_complexity=None, seen_target_freq=None):
    if '⊢ False' in state.pp:
        return 1000000
    if base_complexity is not None:
        return base_complexity + 1
    complexity = 0
    lines = state.pp.split('\n')
    targets = []
    min_freq = 1000000
    for line in lines:
        if line.startswith("⊢"):
            target = line[1:].strip()
            targets.append(target)
            if seen_target_freq is not None:
                if target in seen_target_freq:
                    min_freq = min(min_freq, seen_target_freq[target])
                    seen_target_freq[target] = seen_target_freq[target] + 1
                else:
                    min_freq = 0
                    seen_target_freq[target] = 1
    lengths = [len(x) for x in targets]
    complexity = max(lengths)/2 + sum(lengths)/2
    if seen_target_freq is not None:
        complexity += PENALTY_SEEN_TARGET_MULTIPLIER*min_freq
    return complexity

#@profile
def explore_states(dojo,
                   state_0,
                   theorem,
                   model_state,
                   tokenizer,
                   rag_index,
                   all_tacs, 
                   theorem_code,
                   proof_tactics=None,
                   max_steps=MAX_STEPS, 
                   max_time=1200, 
                   exit_on_finish=True,
                   verbose=False):
    start_time = time.time()

    #if verbose: print(state_0.pp)
    unused_vars = []
    if theorem_code is not None:
        tokens = tokenize_lean_tactic(theorem_code)
        tokens = [x for x in tokens if x.strip() != '']
        type_of_item, def_of_item = classify_lean_elements(state_0.pp)
        unused_vars = [x for x in type_of_item.keys() if x not in tokens]
        #print(unused_vars)

    state_queue = PriorityQueue() # PQ of states, priority being complexity of the state
    state_dict = {} # Keyed by state.pp (or state if it is ProofFinished). Value is (state, its parent states, and tactics from parent state to this state)

    curr_state = state_0
    base_complexity = 0
    state_queue.push(curr_state, explore_state_complexity(curr_state, base_complexity=base_complexity) + randint(0, 4))
    base_complexity += 1
    
    # State graph: Key is state.pp (or ProofFinished state). Value is (state, [parent states], [tactic for each parent state], path from state_0 to state)
    state_dict[curr_state.pp] = (curr_state, list(), list(), list())
    
    # Follow traced_tactics and populate state_dict with the authoratative proof
    if proof_tactics is not None:
        for tactic in proof_tactics:
            try:
                test_state = dojo_run_tac(dojo, curr_state, tactic)
            except:
                break
            if type(test_state) in [LeanError,TimeoutError,TacticResult,DojoCrashError,DojoHardTimeoutError,DojoInitError,ProofGivenUp]:
                break
            elif type(test_state) == lean_dojo.interaction.dojo.ProofFinished:
#                 print("tac1", [tactic])
                state_dict[test_state] = (test_state, [curr_state], [tactic], state_dict[curr_state.pp][3] + [tactic])
                print("Successfully followed proof")
                break
            else:
#                 print("CURR_STATE", curr_state)
#                 print("CURR_STATE_2", state_dict[curr_state.pp])
#                 print("tac_sd", [tactic])
                state_dict[test_state.pp] = (test_state, [curr_state], [tactic], state_dict[curr_state.pp][3] + [tactic]) #store shortest path from state_0
                curr_state = test_state
    n_steps = 0
    theorem_proven = False
    while state_queue.size() > 0:
        curr_state = state_queue.pop()
        if not hasattr(curr_state, 'pp'):
            continue
        #print("POP STATE: ", base_complexity, curr_state.pp, "\nEND STATE")
        type_of_item, def_of_item = classify_lean_elements(curr_state.pp)
        for var in unused_vars:
            if var in type_of_item:
                del type_of_item[var]
            if var in def_of_item:
                del def_of_item[var]
        if verbose:
            if n_steps == 0:
                print("INIT_STATE:", curr_state.pp)
                print(type_of_item)
        #
        #TODO: Add {'nvar0': 'nvar0', 'nvar1': 'nvar1', ...} to type_of_item
        #
        tactic_set = set()
        query = theorem_code + ' # ' + curr_state.pp
        suggestions = get_similar_tacs(query, model_state, tokenizer, rag_index, all_tacs, num_returned=200)
        tac_templates = [x[0] for x in suggestions]
        #
        # Add inverse tactics of any 'rw' tactics in the set
        inv_tac_templates = []
        for tac_template in tac_templates:
            inv_template = get_inverse_tactic(tac_template)
            if inv_template is not None and inv_template not in tac_templates:
                tac_templates.append(inv_template)
        #
        for tac_template in tac_templates:
            try:
                tactics = generate_tactics_from_template(tac_template, type_of_item)
                if len(tactics) > MAX_TACTIC_FROM_TEMPLATE:
                    tactics = random.sample(tactics, MAX_TACTIC_FROM_TEMPLATE)
            except:
                #print("Cannot generate tactics using", tac_template, ":", type_of_item)
                continue
            for tactic in tactics:
                tactic_set.add(tactic)
        #
        proof_finished = False
        tactic_trace = None
        #print(len(tactic_set), "tactics")
        #print(tactic_set)
        for tactic in tactic_set:
            n_steps += 1
            if n_steps % 10000 == 0:
                print(f"{n_steps} steps executed")
            if n_steps > max_steps:
                break
            num_attempt = 0
            while num_attempt < MAX_NUM_DOJO_ATTEMPT:
#                 print("cur", curr_state)
                try:
                    test_state = dojo_run_tac(dojo, curr_state, tactic)
                    break
                except Exception as e:
                    num_attempt += 1
            if num_attempt == MAX_NUM_DOJO_ATTEMPT:
                continue
            #print("TRY TAC:", tactic)
            if type(test_state) in [LeanError,TimeoutError,TacticResult,DojoCrashError,DojoHardTimeoutError,DojoInitError,ProofGivenUp]:
#                 print("err", test_state)
                continue
            elif type(test_state) == lean_dojo.interaction.dojo.ProofFinished:
                proof_finished = True
                if verbose and not theorem_proven:
                    print("ProofFinished")
                if test_state in state_dict:
                    _, parent_states, tactics, prefix = state_dict[test_state]
#                     print("tac3", tactics + [tactic])
                    prefix += [tactic]
                    state_dict[test_state] = (test_state, parent_states + [curr_state], tactics + [tactic], prefix)
#                     print("tacs", state_dict[test_state.pp][-2])
#                     print("prefix", state_dict[test_state.pp][-1])
                    prefix_2 = state_dict[curr_state.pp][3] + [tactic]
                    if len(prefix_2) < len(prefix):
                        state_dict[test_state][3] = (test_state, parent_states + [curr_state], tactics + [tactic], prefix_2)
                else:
                    complexity = 100000000
                    state_queue.push(test_state, complexity)
#                     print("tac4", [tactic])
                    state_dict[test_state] = (test_state, [curr_state], [tactic], state_dict[curr_state.pp][3] + [tactic])
                if exit_on_finish: break
            else:
                #print("TAC:", tactic, "STATE:", test_state.pp)
                if test_state.pp in state_dict:
                    _, parent_states, tactics, prefix = state_dict[test_state.pp]
#                     print("tac5", tactics + [tactic])
#                     print("pref", prefix)
                    prefix += [tactic]
#                     print("tactics", tactics, [tactic], prefix)
                    try:
                        tmp_state = dojo_run_tac(dojo, test_state, tactic)
                        a = tmp_state.pp
                    except Exception as e:
                        continue
                    state_dict[test_state.pp] = (test_state, parent_states + [curr_state], tactics + [tactic], prefix)
                    print("state2", test_state.pp)
                    print("tacs2", state_dict[test_state.pp][-2])
                    print("prefix2", state_dict[test_state.pp][-1])
#                     print("tac", tactic)
#                     print("\n")
#                     prefix_2 = state_dict[curr_state.pp][3] + [tactic]
                    prefix_2 = state_dict[curr_state.pp][3] + [tactic]
                    if len(prefix_2) < len(prefix):
#                         print("prefix3", prefix_2)
                        state_dict[test_state.pp] = (test_state, parent_states + [curr_state], tactics + [tactic], prefix_2)
                else:
                    complexity = explore_state_complexity(test_state, base_complexity=base_complexity)
                    base_complexity += 1
                    state_queue.push(test_state, complexity  + randint(0, 4))
#                     print("tac6", [tactic])
                    print("state3", test_state.pp)
                    print("tacs3", [tactic])
                    print("prefix3", state_dict[curr_state.pp][3] + [tactic])
                    state_dict[test_state.pp] = (test_state, [curr_state], [tactic], state_dict[curr_state.pp][3] + [tactic])
        #
        if proof_finished:
            theorem_proven = True
            if exit_on_finish: break
        if n_steps > max_steps or (not theorem_proven and n_steps > max_steps / 3):
            break
        cur_time = time.time()
        if (cur_time - start_time) > max_time:
            print("Max run-time exceeded")
            break
    
    #     try:
    #         print("Dojo exiting ...")
    #         dojo.__exit__(None, None, None)
    #         print("Dojo exited")
    #         shutil.rmtree(dojo.tmp_dir)
    #         print(dojo.tmp_dir, "removed successfully.")
    #     except Exception as e:
    #         print(f"An error occurred when removing tmp dir: {e}")
    return state_dict, theorem_proven, tactic_trace

file_path = 'Mathlib/Algebra/Polynomial/Derivative.lean'
full_name = 'Polynomial.derivative_C_mul_X_pow'
theorem = Theorem(repo, file_path, full_name)
# traced_file = traced_repo.get_traced_file(file_path)
# thm = traced_file.get_traced_theorem(full_name)
# for premise in traced_file.get_premise_definitions():
#     if premise['full_name'] == full_name:
#         theorem_code = premise['code']
theorem_code = "theorem (a b c : ℝ) : a * (b * c) = b * (a * c)"

# traced_theorems = train_traced_theorems[file_path]
# thm = traced_theorems[full_name]
# theorem_code = thm.comments[0]
# traced_tactics = thm.get_traced_tactics()
# print("traced_tactics:", traced_tactics)

theorem = Theorem(repo, file_path, full_name)

print(theorem_code)

random.seed(42)
dojo, state_0 = Dojo(theorem).__enter__()

# state_dict, theorem_proven, tactic_trace = \
#     explore_states(dojo,
#                    state_0,
#                    theorem,
#                    model_state,
#                    tokenizer,
#                    index,
#                    all_tacs,
#                    theorem_code=theorem_code, 
#                    proof_tactics=None,
#                    max_steps = 10000,
#                    exit_on_finish=False,
#                    verbose=True)

# print(theorem_proven)
# print(tactic_trace)

2024-07-02 14:57:05.792 | WARNING  | lean_dojo.interaction.dojo:__init__:156 - Using Lean 4 without a hard timeout may hang indefinitely.
2024-07-02 14:57:05.793 | INFO     | lean_dojo.interaction.dojo:__enter__:168 - Initializing Dojo for Theorem(repo=LeanGitRepo(url='https://github.com/leanprover-community/mathlib4', commit='27c6744e1c0e25d676be5eb252cd4b6d30c6acc7'), file_path=PosixPath('Mathlib/Algebra/Polynomial/Derivative.lean'), full_name='Polynomial.derivative_C_mul_X_pow')
2024-07-02 14:57:05.795 | INFO     | lean_dojo.interaction.dojo:__enter__:187 - Copy tree ...


theorem (a b c : ℝ) : a * (b * c) = b * (a * c)
get_traced_repo_path, path= /home/mcwave/.cache/lean_dojo/leanprover-community-mathlib4-27c6744e1c0e25d676be5eb252cd4b6d30c6acc7/mathlib4
The traced repo is available in the cache.


2024-07-02 14:57:09.092 | INFO     | lean_dojo.interaction.dojo:_modify_file:385 - Modifying Mathlib/Algebra/Polynomial/Derivative.lean
2024-07-02 14:57:09.095 | INFO     | lean_dojo.interaction.dojo:_modify_file:407 - _modify_file: proof modified
2024-07-02 14:57:09.096 | INFO     | lean_dojo.interaction.dojo:_modify_file:425 - Creating Lean4Repl.lean at Lean4Repl.lean
2024-07-02 14:57:09.096 | INFO     | lean_dojo.interaction.dojo:_modify_file:434 - _modify_file: Done writing to Lean4Repl.lean
2024-07-02 14:57:09.097 | INFO     | lean_dojo.interaction.dojo:_modify_file:439 - _modify_file: All done. Writing to /tmp/tmpn9ntupeg/mathlib4/Mathlib/Algebra/Polynomial/Derivative.lean
2024-07-02 14:57:09.097 | INFO     | lean_dojo.interaction.dojo:__enter__:207 - lake build Lean4Repl ...
2024-07-02 14:57:09.097 | INFO     | lean_dojo.interaction.dojo:__enter__:210 - Launching the proof using <class 'lean_dojo.container.NativeContainer'>
2024-07-02 14:57:09.098 | INFO     | lean_dojo.containe

Post processing ...
Returning ...


In [90]:
current_state = state_dict["m n a b c d : ℕ\n⊢ a ≡ b [MOD n] ↔ ↑n ∣ ↑b - ↑a"][0]

In [25]:
current_state = state_0

In [31]:
state_1 = dojo.run_tac(current_state, "rw [eq_comm]")

print(state_1)
state_1.pp

TacticState(pp='R : Type u\nS : Type v\nT : Type w\nι : Type y\nA : Type z\na✝ b : R\nn✝ : ℕ\ninst✝ : Semiring R\na : R\nn : ℕ\n⊢ C (a * ↑n) * X ^ (n - 1) = derivative (C a * X ^ n)', id=5, message='')


'R : Type u\nS : Type v\nT : Type w\nι : Type y\nA : Type z\na✝ b : R\nn✝ : ℕ\ninst✝ : Semiring R\na : R\nn : ℕ\n⊢ C (a * ↑n) * X ^ (n - 1) = derivative (C a * X ^ n)'

In [15]:
# all_states = list(state_dict.values())
# print(len(all_states), "states in total")
# proven_states = [x[0] for x in all_states if type(x[0]) == lean_dojo.interaction.dojo.ProofFinished]
# print(len(proven_states), "proven states")

# get_tactic_trace(proven_states[0], state_dict)

In [26]:
parent_1 = state_dict[proven_states[0]]
print(parent_1)
print()
parent_2 = state_dict[parent_1[1][0].pp]
print(parent_2)

print()
parent_3 = state_dict[parent_2[1][0].pp]
print(parent_3)

In [97]:
keys = list(state_dict.keys())

for k in range(len(keys)):
    val = state_dict[keys[k]]
    (state, parent_states, tacs) = val
    if 'case' in str(keys[k]):
        continue
    for i in range(len(tacs)):
        if 'mul_left_comm' in tacs[i]:
            print(k)
            print(keys[k])
            print(parent_states[i])
            print(tacs[i])
            print()

1056
R : Type u
inst : CommSemiring R
x y z w w_1 : R
h_1 : w * x + y * (w_1 * z) = 1
⊢ IsCoprime x y
TacticState(pp='R : Type u\ninst : CommSemiring R\nx y z w w_1 : R\nh_1 : w * x + w_1 * (y * z) = 1\n⊢ IsCoprime x y', id=271, message='')
rw [mul_left_comm] at h_1

1056
R : Type u
inst : CommSemiring R
x y z w w_1 : R
h_1 : w * x + y * (w_1 * z) = 1
⊢ IsCoprime x y
TacticState(pp='R : Type u\ninst : CommSemiring R\nx y z w w_1 : R\nh_1 : w * x + w_1 * (y * z) = 1\n⊢ IsCoprime x y', id=271, message='')
rw [← mul_left_comm] at h_1



In [4]:
# Check which theorems can be proven

import pickle

file_paths = [
    "MIL/C02_Basics/solutions/Solutions_S01_Calculating.lean",
    "MIL/C02_Basics/solutions/Solutions_S02_Proving_Identities_in_Algebraic_Structures.lean",
    "MIL/C02_Basics/solutions/Solutions_S03_Using_Theorems_and_Lemmas.lean",
    "MIL/C02_Basics/solutions/Solutions_S04_More_on_Order_and_Divisibility.lean",
    "MIL/C02_Basics/solutions/Solutions_S05_Proving_Facts_about_Algebraic_Structures.lean"
]

# theorems_to_compute = [
#     "MyRing.self_sub",
#     "MyRing.one_add_one_eq_two",
#     "MyRing.two_mul",
#     "MyGroup.mul_right_inv",
#     "MyGroup.mul_one",
#     "MyGroup.mul_inv_rev",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex6",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex7",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex8",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex9",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex10"
# ]

#fout = open('/home/mcwave/code/automath/atp/datasets/provability_raw/C02_Basics_S02part2.pkl', 'wb')

for file_path in file_paths:
    traced_file = traced_repo.get_traced_file(file_path)
    premises = traced_file.get_premise_definitions()
    #
    results = []
    for premise in premises:
#         if file_path != "MIL/C02_Basics/solutions/Solutions_S05_Proving_Facts_about_Algebraic_Structures.lean" and \
#             premise['full_name'] not in theorems_to_compute:
#             continue
        if premise['code'].startswith('theorem '):
            print('THEOREM:', premise['full_name'])
            print(premise['code'])
            #theorem = Theorem(repo, file_path, premise['full_name'])
            thm = traced_file.get_traced_theorem(premise['full_name'])
            theorem = thm.theorem
            print("theorem loaded")
            state_dict, theorem_proven, tactic_trace = \
            explore_states(theorem,
                           index,
                           tacs,
                           theorem_code=premise['code'],
                           max_steps = 20000,
                           exit_on_finish=True,
                           verbose=True)
            print("Proved?:", theorem_proven)
            print(tactic_trace)
                
#fout.close()

NameError: name 'traced_repo' is not defined

In [13]:
# Generate state pairs and their distances. 
# Probably useful if we hope to generate intermediate states (e.g., "have").
from datetime import datetime
from collections import defaultdict

def yield_state_pairs(curr_state, leaf_state, state_dict, seen_states_pp, tactic_list, max_distance=8, max_parents=10):
    #     print("curr_state:", curr_state.pp)
    #     print("leaf_state:", leaf_state.pp)
    if type(curr_state) == lean_dojo.interaction.dojo.ProofFinished:
        _, parent_states, tactics, prefix = state_dict[curr_state]
    else:
        _, parent_states, tactics, prefix = state_dict[curr_state.pp]
    if len(seen_states_pp) > 1:
        yield (curr_state, leaf_state, len(seen_states_pp) - 1, tactic_list)
    if len(seen_states_pp) <= max_distance:
        num_parents_checked = 0
        for i in range(len(parent_states)):
            parent_state = parent_states[i]
            if parent_state.pp not in seen_states_pp:
                yield from yield_state_pairs(parent_state, 
                                             leaf_state, 
                                             state_dict, 
                                             seen_states_pp + [parent_state.pp],
                                             [tactics[i]] + tactic_list
                                            )
                num_parents_checked += 1
                if num_parents_checked >= max_parents:
                    break
    return

@timeout(3000)
def get_state_provability_data(dojo,
                               state_0,
                               theorem,
                               model_state,
                               tokenizer,
                               rag_index, 
                               all_tacs,
                               theorem_code=None, 
                               proof_tactics=None,
                               max_steps=100000, 
                               negative_ratio=1.0,
                               verbose=False):
    MAX_PROVEN_STATES = 1000
    start_time = datetime.now()
    print("Start exploring states at", start_time.time())
    state_dict, theorem_proven, tactic_trace = \
    explore_states(dojo,
                   state_0,
                   theorem,
                   model_state,
                   tokenizer,
                   rag_index, 
                   all_tacs,
                   theorem_code=theorem_code, 
                   proof_tactics=proof_tactics,
                   max_steps=max_steps,
                   exit_on_finish=False,
                   verbose=verbose)
    all_states = list(state_dict.values())
    print(f"{len(all_states)} states in total", datetime.now().time())
    proven_states = [x[0] for x in all_states if type(x[0]) == lean_dojo.interaction.dojo.ProofFinished]
    print(f"{len(proven_states)} proven states")
    state_pairs = []
    seen_state_pps = defaultdict(int)

    random.shuffle(proven_states)
    for i in range(len(proven_states)):
        proven_state = proven_states[i]
        #print(proven_state)
        pairs = list(yield_state_pairs(proven_state, proven_state, state_dict, [str(proven_state)], []))
        ancestor_state_dict = {}
        for ancestor, leaf, distance, tactic in pairs:
            if ancestor.pp not in ancestor_state_dict:
                ancestor_state_dict[ancestor] = (distance, tactic)
            else:
                if distance < ancestor_state_dist[ancestor]:
                    ancestor_state_dict[ancestor] = (distance, tactic)
        #print(ancestor_state_dist)
        #print(len(ancestor_state_dict), "ancestors found")
        for ancestor, (distance, tactic) in ancestor_state_dict.items():
            if seen_state_pps[ancestor.pp] < 10:
                print("anc", state_dict[ancestor.pp])
                state_pairs.append((ancestor, proven_state, distance, tactic, state_dict[ancestor.pp][-1]))
                seen_state_pps[ancestor.pp] += 1
            if (datetime.now() - start_time).seconds > 2900:
                return state_pairs, state_dict
        if i % 100 == 0 and i > 0:
            print(f"{i} proven states processed")
        if i > MAX_PROVEN_STATES:
            break
    #
    num_positive = len(state_pairs)
    # Add negative examples
    if num_positive > 0:
        num_negative = 0
        for tup in all_states:
            state = tup[0]
            if hasattr(state, 'pp') and state.pp not in seen_state_pps:
                state_pairs.append((state, random.choice(proven_states), -1, None, None))
                num_negative += 1
                if num_negative >= num_positive * negative_ratio:
                    break
                if (datetime.now() - start_time).seconds > 2900:
                    return state_pairs, state_dict
    return state_pairs, state_dict

file_path = 'Mathlib/RingTheory/Coprime/Basic.lean'
full_name = 'IsCoprime.of_mul_right_left'
theorem = Theorem(repo, file_path, full_name)
# traced_file = traced_repo.get_traced_file(file_path)
# thm = traced_file.get_traced_theorem(full_name)
# for premise in traced_file.get_premise_definitions():
#     if premise['full_name'] == full_name:
#         theorem_code = premise['code']

# traced_theorems = train_traced_theorems[file_path]
# thm = traced_theorems[full_name]
theorem_code = "theorem IsCoprime.of_mul_right_left (H : IsCoprime x (y * z)) : IsCoprime x y"
# traced_tactics = thm.get_traced_tactics()
# print("traced_tactics:", traced_tactics)

theorem = Theorem(repo, file_path, full_name)

random.seed(42)
dojo, state_0 = Dojo(theorem).__enter__()
print("Generating state pairs")
state_pairs, state_dict = \
    get_state_provability_data(dojo,
                               state_0,
                               theorem,
                               model_state,
                               tokenizer,
                               index,
                               all_tacs,
                               theorem_code=theorem_code,
                               proof_tactics=None,
                               max_steps = 10000,
                               verbose=True)

print(len(state_pairs))

2024-06-30 00:09:43.164 | WARNING  | lean_dojo.interaction.dojo:__init__:156 - Using Lean 4 without a hard timeout may hang indefinitely.
2024-06-30 00:09:43.164 | INFO     | lean_dojo.interaction.dojo:__enter__:168 - Initializing Dojo for Theorem(repo=LeanGitRepo(url='https://github.com/leanprover-community/mathlib4', commit='27c6744e1c0e25d676be5eb252cd4b6d30c6acc7'), file_path=PosixPath('Mathlib/RingTheory/Coprime/Basic.lean'), full_name='IsCoprime.of_mul_right_left')
2024-06-30 00:09:43.165 | INFO     | lean_dojo.interaction.dojo:__enter__:187 - Copy tree ...


get_traced_repo_path, path= /home/mcwave/.cache/lean_dojo/leanprover-community-mathlib4-27c6744e1c0e25d676be5eb252cd4b6d30c6acc7/mathlib4
The traced repo is available in the cache.


2024-06-30 00:10:31.447 | INFO     | lean_dojo.interaction.dojo:_modify_file:385 - Modifying Mathlib/RingTheory/Coprime/Basic.lean
2024-06-30 00:10:31.450 | INFO     | lean_dojo.interaction.dojo:_modify_file:407 - _modify_file: proof modified
2024-06-30 00:10:31.451 | INFO     | lean_dojo.interaction.dojo:_modify_file:425 - Creating Lean4Repl.lean at Lean4Repl.lean
2024-06-30 00:10:31.451 | INFO     | lean_dojo.interaction.dojo:_modify_file:434 - _modify_file: Done writing to Lean4Repl.lean
2024-06-30 00:10:32.464 | INFO     | lean_dojo.interaction.dojo:_modify_file:439 - _modify_file: All done. Writing to /tmp/tmp4ecydwdy/mathlib4/Mathlib/RingTheory/Coprime/Basic.lean
2024-06-30 00:10:32.465 | INFO     | lean_dojo.interaction.dojo:__enter__:207 - lake build Lean4Repl ...
2024-06-30 00:10:32.466 | INFO     | lean_dojo.interaction.dojo:__enter__:210 - Launching the proof using <class 'lean_dojo.container.NativeContainer'>
2024-06-30 00:10:32.466 | INFO     | lean_dojo.container:run:181 

Post processing ...
Returning ...
Generating state pairs
Start exploring states at 00:15:26.546946
INIT_STATE: R : Type u
inst✝ : CommSemiring R
x y z : R
H : IsCoprime x (y * z)
⊢ IsCoprime x y
{'x': 'variable', 'y': 'variable', 'z': 'variable', 'H': 'unknown'}
ProofFinished
10000 steps executed
383 states in total 00:15:54.991433
1 proven states
anc (TacticState(pp='R : Type u\ninst✝ : CommSemiring R\nx y z : R\nH : IsCoprime (y * z) x\n⊢ IsCoprime y x', id=807, message=''), [TacticState(pp='R : Type u\ninst✝ : CommSemiring R\nx y z : R\nH : IsCoprime x (y * z)\n⊢ IsCoprime x y', id=0, message=None), TacticState(pp='R : Type u\ninst✝ : CommSemiring R\nx y z : R\nH : IsCoprime x (y * z)\n⊢ IsCoprime x y', id=0, message=None), TacticState(pp='R : Type u\ninst✝ : CommSemiring R\nx y z : R\nH : IsCoprime (y * z) x\n⊢ IsCoprime y x', id=8, message=''), TacticState(pp='R : Type u\ninst✝ : CommSemiring R\nx y z : R\nH : IsCoprime (y * z) x\n⊢ IsCoprime y x', id=8, message=''), TacticState(p

In [43]:
import pickle
import os

def get_filename(filepath):
    # Extract the base name (e.g., 'example.txt' from '/path/to/example.txt')
    base_name = os.path.basename(filepath)
    # Split the base name and the extension and return just the base name
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

file_paths = [
    "MIL/C02_Basics/solutions/Solutions_S01_Calculating.lean",
    "MIL/C02_Basics/solutions/Solutions_S02_Proving_Identities_in_Algebraic_Structures.lean",
    "MIL/C02_Basics/solutions/Solutions_S03_Using_Theorems_and_Lemmas.lean",
    "MIL/C02_Basics/solutions/Solutions_S04_More_on_Order_and_Divisibility.lean",
    "MIL/C02_Basics/solutions/Solutions_S05_Proving_Facts_about_Algebraic_Structures.lean"
]

# theorems_to_compute = [
#     "MyRing.self_sub",
#     "MyRing.one_add_one_eq_two",
#     "MyRing.two_mul",
#     "MyGroup.mul_right_inv",
#     "MyGroup.mul_one",
#     "MyGroup.mul_inv_rev",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex6",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex7",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex8",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex9",
#     "C02S04.Solutions_S04_More_on_Order_and_Divisibility_ex10"
# ]

def compute_provability_training_data(file_path, output_path):
    file_name = get_filename(file_path)
    fout = open(output_path + file_name + '.pkl', 'wb')
    traced_file = traced_repo.get_traced_file(file_path)
    premises = traced_file.get_premise_definitions()
    #
    results = []
    for premise in premises:
        if premise['code'].startswith('theorem '):
            print('THEOREM:', premise['full_name'])
            print(premise['code'])
            thm = traced_file.get_traced_theorem(premise['full_name'])
            theorem = thm.theorem
            traced_statistic = thm.get_traced_tactics()
            print("theorem loaded")
            state_pairs, _ = \
                get_state_provability_data(theorem,
                                           index,
                                           tacs,
                                           theorem_code=premise['code'],
                                           traced_tactics=traced_tactics,
                                           max_steps=10000,
                                           verbose=True)
            print(len(state_pairs), "state pairs found")
            #
            for state_pair in state_pairs:
                result = (file_path, premise, theorem, state_pair)
                results.append(result)
                pickle.dump(result, fout)
    #
    fout.close()
    return len(results)
    
    
OUTPUT_FOLDER = '/home/mcwave/code/automath/atp/datasets/provability/rag/'

for file_path in file_paths:
    compute_provability_training_data(file_path, OUTPUT_FOLDER)

2024-06-03 22:17:53.707 | WARNING  | lean_dojo.interaction.dojo:__init__:162 - Using Lean 4 without a hard timeout may hang indefinitely.


THEOREM: Solutions_S01_Calculating_ex1
theorem Solutions_S01_Calculating_ex1 (a b c : ℝ) : c * b * a = b * (a * c)
theorem loaded
INIT_STATE: a b c : ℝ
⊢ c * b * a = b * (a * c)
{'a': 'variable', 'b': 'variable', 'c': 'variable'}
ProofFinished
ProofFinished
10000 steps executed
1045 states in total
44 proven states
ProofFinished(tactic_state_id=28, message='')
15 ancestors found
ProofFinished(tactic_state_id=35, message='')
15 ancestors found
ProofFinished(tactic_state_id=93, message='')
15 ancestors found
ProofFinished(tactic_state_id=100, message='')
15 ancestors found
ProofFinished(tactic_state_id=131, message='')
15 ancestors found
ProofFinished(tactic_state_id=138, message='')
15 ancestors found
ProofFinished(tactic_state_id=236, message='')
15 ancestors found
ProofFinished(tactic_state_id=242, message='')
15 ancestors found
ProofFinished(tactic_state_id=305, message='')
15 ancestors found
ProofFinished(tactic_state_id=311, message='')
15 ancestors found
ProofFinished(tactic_state

2024-06-03 22:18:50.442 | WARNING  | lean_dojo.interaction.dojo:__init__:162 - Using Lean 4 without a hard timeout may hang indefinitely.


15 ancestors found
ProofFinished(tactic_state_id=1566, message='')
15 ancestors found
ProofFinished(tactic_state_id=1602, message='')
15 ancestors found
ProofFinished(tactic_state_id=1609, message='')
15 ancestors found
ProofFinished(tactic_state_id=1720, message='')
15 ancestors found
ProofFinished(tactic_state_id=1726, message='')
15 ancestors found
1408 state pairs found
THEOREM: Solutions_S01_Calculating_ex2
theorem Solutions_S01_Calculating_ex2 (a b c : ℝ) : a * (b * c) = b * (a * c)
theorem loaded


SystemExit: -1

/home/mcwave/anaconda3/envs/atp/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [138]:
import ray

# Initialize Ray cautiously
if not ray.is_initialized():
    ray.init(num_cpus=5)

@ray.remote
def compute_provability_training_data_remote(file_path, output_path):
    compute_provability_training_data(file_path, output_path)

# Submit tasks
result_ids = [compute_provability_training_data_remote.remote(i) for file_path in file_paths]

# Fetch results
results = ray.get(result_ids)

# Optionally, shut down Ray if you're done with all computations
ray.shutdown()

# Display results
print(results)

2024-06-02 22:23:43,060	INFO worker.py:1740 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 
(raylet) [2024-06-02 22:24:43,027 E 313863 313863] (raylet) node_manager.cc:3002: 3 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 2c0ea0932736f0540f910b5bb7c16183436a30e051a7aa0d64e2e589, IP: 192.168.0.192) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 192.168.0.192`
(raylet) 
(raylet) Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero

ValueError: The remote function __main__.compute_provability_training_data_remote is too large (2926 MiB > FUNCTION_SIZE_ERROR_THRESHOLD=95 MiB). Check that its definition is not implicitly capturing a large array or other object in scope. Tip: use ray.put() to put large objects in the Ray object store.

(raylet) [2024-06-02 22:25:43,027 E 313863 313863] (raylet) node_manager.cc:3002: 2 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 2c0ea0932736f0540f910b5bb7c16183436a30e051a7aa0d64e2e589, IP: 192.168.0.192) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 192.168.0.192`
(raylet) 
(raylet) Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.


In [19]:
# Create the inverse tactic for a 'rw' tactic template
def get_inverse_tactic(tac_template):
    if not tac_template.startswith('rw') or not '[' in tac_template \
        or '←' in tac_template or '[{' in tac_template:
        return None
    result = tac_template.replace('[', '[← ')
    if ',' in result:
        pos = result.find(',')
        result = result[0:pos] + ']'
    return result

for tac, _ in suggestions:
    print(tac)
    print(get_inverse_tactic(tac))
    print()

rw [mul_comm]
rw [← mul_comm]

rw [mul_assoc]
rw [← mul_assoc]

ring
None

rw [mul_one]
rw [← mul_one]

linarith
None

subst {unknown}
None

> rfl
None

abel
None

norm_cast
None

field_simp
None

norm_num
None

> simp
None

cases {unknown} with
None

symm
None

gcongr
None

positivity
None

simp
None

calc
None

cases {unknown}
None

contradiction
None

rw [mul_add]
rw [← mul_add]

rfl
None

simp_all
None

constructor
None

congr
None

intro
None

simpa
None

rw [← {unknown}]
None

classical
None

simp [*]
None

simp only [{unknown}]
None

ext
None

exfalso
None

rwa [{unknown}]
None

split
None

simp [{unknown}]
None

rw [mul_zero]
rw [← mul_zero]

rw [{unknown}]
None

rw[{unknown}]
None

apply {unknown}
None

>
None

tauto
None

rw [one_mul]
rw [← one_mul]

congr 1
None

dsimp
None

exact {unknown}
None

intros
None

dsimp only
None

push_cast
None

decide
None

by_contra! {nvar0}
None

aesop
None

swap
None

left
None

rw [{hypothesis}]
None

rintro rfl
None

exact
None

rw [{funct

In [5]:
import atexit

import posix_ipc
import time
import threading

class SystemSemaphore:
    def __init__(self, name, limit, max_hold_time=240, acquire_timeout=60):
        self.name = name
        self.limit = limit
        self.max_hold_time = max_hold_time
        self.acquire_timeout = acquire_timeout

    def __enter__(self):
        self.lock = posix_ipc.Semaphore(f'/{self.name}', posix_ipc.O_CREAT, 0x384, self.limit)
        start_time = time.time()
        while True:
            if self.lock.acquire(self.acquire_timeout):
                break
            elapsed_time = time.time() - start_time
            if elapsed_time > self.acquire_timeout:
                raise TimeoutError(f"Failed to acquire semaphore within {self.acquire_timeout} seconds.")

        self.start_time = time.time()
        self.timer = threading.Timer(self.max_hold_time, self.release_semaphore)
        self.timer.start()

    def __exit__(self, _type, value, tb):
        self.timer.cancel()
        self.release_semaphore()

    def release_semaphore(self):
        elapsed_time = time.time() - self.start_time
        if elapsed_time > self.max_hold_time:
            print(f"Warning: Semaphore was held for {elapsed_time:.2f} seconds, exceeding the maximum hold time of {self.max_hold_time} seconds.")
        self.lock.release()

    def unlink(self):
        posix_ipc.unlink_semaphore(f'/{self.name}')

semaphore = SystemSemaphore('dojo-enter-1', 1)

def cleanup():
    semaphore.unlink()

atexit.register(cleanup)

semaphore.unlink()

In [9]:
log_file_path = '/home/mcwave/code/automath/atp/datasets/mathlib4_rag_explore_state_log.txt'

import re

def parse_rag_explore_state_log(log_file_path):
    output = []
    # Regular expression pattern to match PosixPath and full_name
    pattern = r"PosixPath\('(.+?)'\).+full_name='(.+?)'"
    # Open the text file for reading
    with open(log_file_path, 'r') as file:
        # Read the file line by line
        for line in file:
            if 'Start entering ' not in line:
                continue
            # Use re.search to find the matches
            match = re.search(pattern, line)
            if match:
                posix_path = match.group(1)
                full_name = match.group(2)
            else:
                continue
            output.append((posix_path, full_name))
    return output


output = parse_rag_explore_state_log(log_file_path)
print(len(output), "theorems processed")
output

261 theorems processed


[('.lake/packages/mathlib/Mathlib/RingTheory/Coprime/Basic.lean',
  'isCoprime_self'),
 ('.lake/packages/mathlib/Mathlib/Algebra/Group/Conj.lean',
  'ConjClasses.map_surjective'),
 ('.lake/packages/mathlib/Mathlib/Data/Nat/Squarefree.lean',
  'Nat.squarefree_iff_nodup_factors'),
 ('.lake/packages/mathlib/Mathlib/GroupTheory/GroupAction/Prod.lean',
  'Prod.smul_zero_mk'),
 ('.lake/packages/mathlib/Mathlib/Algebra/GroupWithZero/Divisibility.lean',
  'mul_dvd_mul_iff_left'),
 ('.lake/packages/mathlib/Mathlib/GroupTheory/Perm/Sign.lean',
  'Equiv.Perm.perm_inv_on_of_perm_on_finset'),
 ('.lake/packages/mathlib/Mathlib/RingTheory/Coprime/Basic.lean',
  'isCoprime_zero_left'),
 ('.lake/packages/mathlib/Mathlib/Data/Nat/Squarefree.lean',
  'Squarefree.natFactorization_le_one'),
 ('.lake/packages/mathlib/Mathlib/Algebra/Group/Conj.lean',
  'isConj_iff_conjugatesOf_eq'),
 ('.lake/packages/mathlib/Mathlib/Algebra/Group/Conj.lean',
  'ConjClasses.mem_carrier_iff_mk_eq'),
 ('.lake/packages/mathlib/

In [29]:
import pickle

results = []
#fin = open('/home/mcwave/code/automath/atp/datasets/provability/rag/mathlib__Mathlib__RingTheory__Coprime__Basic.lean.pkl', 'rb')
fin = open('/home/mcwave/code/automath/atp/datasets/provability/rag/Mathlib__Analysis__Calculus__FDeriv__Star.lean.pkl', 'rb')
while True:
    try:
        result = pickle.load(fin)
    except Exception as e:
        print(e)
        break
    file_path, full_name, theorem, state_pair = result
    results.append(result)

fin.close()

Ran out of input


In [31]:
len(results)

13073

In [21]:
theorems = set()
proofs = {}
for file_path, full_name, theorem, state_pair in results:
    theorems.add(full_name)
    if full_name not in proofs:
        proofs[full_name] = []
print(theorems)

{'Polynomial.derivative_sum', 'Polynomial.natDegree_derivative_lt', 'Polynomial.derivative_comp', 'Polynomial.derivative_add', 'Polynomial.iterate_derivative_neg', 'Polynomial.iterate_derivative_eq_zero', 'Polynomial.derivative_ofNat', 'Polynomial.derivative_apply', 'Polynomial.derivative_eval', 'Polynomial.iterate_derivative_intCast_mul', 'Polynomial.derivative_X_sub_C', 'Polynomial.iterate_derivative_C', 'Polynomial.derivative_X_add_C_pow', 'Polynomial.derivative_sub', 'Polynomial.eq_C_of_derivative_eq_zero', 'Polynomial.iterate_derivative_X_pow_eq_natCast_mul', 'Polynomial.derivative_X', 'Polynomial.iterate_derivative_X_pow_eq_C_mul', 'Polynomial.iterate_derivative_one', 'Polynomial.dvd_iterate_derivative_pow', 'Polynomial.degree_derivative_le', 'Polynomial.iterate_derivative_zero', 'Polynomial.derivative_pow', 'Polynomial.derivative_C_mul_X_pow', 'Polynomial.derivative_sq', 'Polynomial.derivative_natCast_mul', 'Polynomial.derivative_pow_succ', 'Polynomial.derivative_prod', 'Polynom

In [35]:
results[100]

('Mathlib/Analysis/Calculus/FDeriv/Star.lean',
 'DifferentiableAt.star',
 Theorem(repo=LeanGitRepo(url='https://github.com/leanprover-community/mathlib4', commit='27c6744e1c0e25d676be5eb252cd4b6d30c6acc7'), file_path=PosixPath('Mathlib/Analysis/Calculus/FDeriv/Star.lean'), full_name='DifferentiableAt.star'),
 (TacticState(pp="case neg\n𝕜 : Type u_1\ninst✝⁹ : NontriviallyNormedField 𝕜\ninst✝⁸ : StarRing 𝕜\ninst✝⁷ : TrivialStar 𝕜\nE : Type u_2\ninst✝⁶ : NormedAddCommGroup E\ninst✝⁵ : NormedSpace 𝕜 E\nF : Type u_3\ninst✝⁴ : NormedAddCommGroup F\ninst✝³ : StarAddMonoid F\ninst✝² : NormedSpace 𝕜 F\ninst✝¹ : StarModule 𝕜 F\ninst✝ : ContinuousStar F\nf : E → F\nf' e : E →L[𝕜] F\nx : E\ns : Set E\nL : Filter E\nh : DifferentiableAt 𝕜 f x\nnvar0 : ¬x ≠ 0\n⊢ DifferentiableAt 𝕜 (fun y => Star.star (f y)) x\n\ncase pos\n𝕜 : Type u_1\ninst✝⁹ : NontriviallyNormedField 𝕜\ninst✝⁸ : StarRing 𝕜\ninst✝⁷ : TrivialStar 𝕜\nE : Type u_2\ninst✝⁶ : NormedAddCommGroup E\ninst✝⁵ : NormedSpace 𝕜 E\nF : Type u_3\n

In [8]:
def is_pp_useful(pp):
    lines = pp.split('\n')
    targets = []
    conditions = []
    min_freq = 1000000
    for line in lines:
        if line.startswith("⊢"):
            target = line.strip()
            targets.append(target)
        else:
            conditions.append(line.strip())
    if '⊢ False' in targets:
        return False
    if '⊢ R' in targets:
        return False
    if '?m.' in pp:
        return False
    return True

num_output = 0
for i in range(len(results)):
    file_path, full_name, theorem, state_pair = results[i]
    if full_name != 'Nat.mem_primeFactors_of_ne_zero': #'IsCoprime.of_mul_right_left': #'IsCoprime.mul_right':
        continue
    if not is_pp_useful(results[i][3][0].pp):
        continue
    print("CASE", i)
    print(results[i][3][0].pp)
    print(theorem)
    print()
    num_output += 1
    if num_output > 400:
        break

CASE 0
a b k m n p : ℕ
hn : n ≠ 0
⊢ p ∈ n.primeFactors ↔ Prime p ∧ p ∣ n
Theorem(repo=LeanGitRepo(url='https://github.com/xiaoxin-yin/math-in-lean', commit='20077bcd4392317ddb9605404fda3a85e40e8956'), file_path=PosixPath('.lake/packages/mathlib/Mathlib/Data/Nat/PrimeFin.lean'), full_name='Nat.mem_primeFactors_of_ne_zero')

CASE 1
a b k m n p : ℕ
hn : n ≠ 0
⊢ p ∈ n.primeFactors ↔ _root_.Prime p ∧ p ∣ n
Theorem(repo=LeanGitRepo(url='https://github.com/xiaoxin-yin/math-in-lean', commit='20077bcd4392317ddb9605404fda3a85e40e8956'), file_path=PosixPath('.lake/packages/mathlib/Mathlib/Data/Nat/PrimeFin.lean'), full_name='Nat.mem_primeFactors_of_ne_zero')

CASE 2
a b k m n p : ℕ
hn : n ≠ 0
⊢ p ∈ n.primeFactors ↔ Irreducible p ∧ p ∣ n
Theorem(repo=LeanGitRepo(url='https://github.com/xiaoxin-yin/math-in-lean', commit='20077bcd4392317ddb9605404fda3a85e40e8956'), file_path=PosixPath('.lake/packages/mathlib/Mathlib/Data/Nat/PrimeFin.lean'), full_name='Nat.mem_primeFactors_of_ne_zero')

CASE 3
a b k